In [1]:
import os
import glob
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import h5py
import scipy
from scipy.sparse import csr_matrix
import yaml
import pyranges as pr

import anndata as an
import scanpy as sc

# Load the GTF

In [44]:
%%time

# Load GTF
gtf_path = "/nfs/turbo/umms-indikar/shared/projects/reference_genome/prebuilt/refdata-gex-GRCh38-2024-A/genes/genes.gtf"
print(f"Loading GTF from: {gtf_path}")
gtf = pr.read_gtf(gtf_path).as_df()
print(f"Total entries in GTF: {len(gtf)}")

# Transcript-level annotations
transcripts = gtf[gtf.Feature == "transcript"]
transcripts = transcripts[[
    "gene_id", "gene_name", "gene_type",
    "transcript_id", "transcript_name", "transcript_type",
    "Chromosome", "Start", "End"
]].dropna(subset=["gene_name", "transcript_name"]).drop_duplicates().reset_index(drop=True)
print(f"Transcripts loaded: {len(transcripts)}")

# Gene-level annotations
genes = gtf[gtf.Feature == "gene"]
genes = genes[[
    "gene_id", "gene_name", "gene_type",
    "Chromosome", "Start", "End"
]].dropna(subset=["gene_name"]).drop_duplicates().reset_index(drop=True)
print(f"Genes loaded: {len(genes)}")

Loading GTF from: /nfs/turbo/umms-indikar/shared/projects/reference_genome/prebuilt/refdata-gex-GRCh38-2024-A/genes/genes.gtf
Total entries in GTF: 3293161
Transcripts loaded: 226005
Genes loaded: 38606
CPU times: user 59.4 s, sys: 6.16 s, total: 1min 5s
Wall time: 1min 5s


# load the gene anndatas

In [55]:
%%time

outpath = "/nfs/turbo/umms-indikar/shared/projects/lab_data/genes.raw.h5ad"

hsc_subdirs = ['fast_v3', 'minion_384', 'fast_v4.2', 'hac_v4.2', 'hac_v4.3', 'hac_v3']

fpaths = {
    'hybrid' : "/nfs/turbo/umms-indikar/shared/projects/hybrid_reprogramming/pipeline_outputs/hyb_epi2me_final/hybrid/hybrid.gene_raw_feature_bc_matrix",
    'cell_cycle' : "/nfs/turbo/umms-indikar/shared/projects/HSC/pipeline_outputs/cc_fibroblast_full/scfib/scfib.gene_raw_feature_bc_matrix",
    'hsc' : "/nfs/turbo/umms-indikar/shared/projects/HSC/pipeline_outputs/hsc_epi2me_full/",
}

# Load and tag
adatas = []
for label, path in fpaths.items():
    if not label == 'hsc':
        adata = sc.read_10x_mtx(path, var_names='gene_symbols', cache=True)
        adata.obs['dataset'] = label
        adata.obs_names = [f"{label}_{x}" for x in adata.obs_names] 
        adatas.append(adata)
        print(f"{label} {adata.shape}")
    else:
        hsc_adatas = []
        for subdir in hsc_subdirs:
            fpath = f"{path}{subdir}/{subdir}.gene_raw_feature_bc_matrix"
            adata = sc.read_10x_mtx(fpath, var_names='gene_symbols', cache=True)
            adata.obs['cell_id'] = adata.obs_names
            hsc_adatas.append(adata)
            print(f"HSC {subdir} {adata.shape}")

        adata = an.concat(hsc_adatas, join='outer', keys=hsc_subdirs)
        adata = sc.get.aggregate(
            adata, 
            by='cell_id',
            func='sum',
        )
        adata.X = adata.layers['sum']
        del adata.layers['sum']
        del adata.obs['cell_id']
        adata.obs['dataset'] = label
        adata.obs_names = [f"{label}_{x}" for x in adata.obs_names]
        adatas.append(adata)

# Concatenate into one AnnData object
adata = an.concat(adatas, join='outer', label='dataset', keys=fpaths.keys())
genes_indexed = genes.drop_duplicates(subset='gene_name').set_index('gene_name')
var_annot = adata.var.join(genes_indexed, how='left')
adata.var = var_annot

adata.write(outpath)
print(f"saved to: {outpath}")
adata

hybrid (10895, 27819)
cell_cycle (8963, 23635)
HSC fast_v3 (9525, 8839)
HSC minion_384 (8583, 18677)
HSC fast_v4.2 (8646, 9358)
HSC hac_v4.2 (8657, 23924)
HSC hac_v4.3 (8587, 18742)
HSC hac_v3 (8576, 22330)


/home/cstansbu/miniconda3/envs/scanpy/lib/python3.12/site-packages/anndata/_core/anndata.py:1818: UserWarning: Observation names are not unique. To make them unique, call `.obs_names_make_unique`.
  utils.warn_names_duplicates("obs")


CPU times: user 11.3 s, sys: 5.15 s, total: 16.5 s
Wall time: 21.7 s


AnnData object with n_obs × n_vars = 31261 × 30032
    obs: 'dataset'
    var: 'gene_id', 'gene_type', 'Chromosome', 'Start', 'End'

# Load the transcripts

In [58]:
%%time

outpath = "/nfs/turbo/umms-indikar/shared/projects/lab_data/transcripts.raw.h5ad"

hsc_subdirs = ['fast_v3', 'minion_384', 'fast_v4.2', 'hac_v4.2', 'hac_v4.3', 'hac_v3']

fpaths = {
    'hybrid' : "/nfs/turbo/umms-indikar/shared/projects/hybrid_reprogramming/pipeline_outputs/hyb_epi2me_final/hybrid/hybrid.transcript_raw_feature_bc_matrix",
    'cell_cycle' : "/nfs/turbo/umms-indikar/shared/projects/HSC/pipeline_outputs/cc_fibroblast_full/scfib/scfib.transcript_raw_feature_bc_matrix",
    'hsc' : "/nfs/turbo/umms-indikar/shared/projects/HSC/pipeline_outputs/hsc_epi2me_full/",
}

# Load and tag
adatas = []
for label, path in fpaths.items():
    if not label == 'hsc':
        adata = sc.read_10x_mtx(path, var_names='gene_symbols', cache=True)
        adata.obs['dataset'] = label
        adata.obs_names = [f"{label}_{x}" for x in adata.obs_names] 
        adatas.append(adata)
        print(f"{label} {adata.shape}")
    else:
        hsc_adatas = []
        for subdir in hsc_subdirs:
            fpath = f"{path}{subdir}/{subdir}.transcript_raw_feature_bc_matrix"
            adata = sc.read_10x_mtx(fpath, var_names='gene_symbols', cache=True)
            adata.obs['cell_id'] = adata.obs_names
            hsc_adatas.append(adata)
            print(f"HSC {subdir} {adata.shape}")

        adata = an.concat(hsc_adatas, join='outer', keys=hsc_subdirs)
        adata = sc.get.aggregate(
            adata, 
            by='cell_id',
            func='sum',
        )
        adata.X = adata.layers['sum']
        del adata.layers['sum']
        del adata.obs['cell_id']
        adata.obs['dataset'] = label
        adata.obs_names = [f"{label}_{x}" for x in adata.obs_names]
        adatas.append(adata)

# Concatenate into one AnnData object
adata = an.concat(adatas, join='outer', label='dataset', keys=fpaths.keys())

transcripts_indexed = transcripts.drop_duplicates(subset='transcript_name').set_index('transcript_id')
var_annot = adata.var.join(transcripts_indexed, how='left')
adata.var = var_annot

adata.write(outpath)
print(f"saved to: {outpath}")
adata

hybrid (10895, 77111)
cell_cycle (8963, 63464)
HSC fast_v3 (9279, 11124)
HSC minion_384 (8579, 39453)
HSC fast_v4.2 (8426, 12175)
HSC hac_v4.2 (8657, 60494)
HSC hac_v4.3 (8579, 39922)
HSC hac_v3 (8576, 53841)


/home/cstansbu/miniconda3/envs/scanpy/lib/python3.12/site-packages/anndata/_core/anndata.py:1818: UserWarning: Observation names are not unique. To make them unique, call `.obs_names_make_unique`.
  utils.warn_names_duplicates("obs")


saved to: /nfs/turbo/umms-indikar/shared/projects/lab_data/transcripts.raw.h5ad
CPU times: user 13.3 s, sys: 5.79 s, total: 19.1 s
Wall time: 22.5 s


AnnData object with n_obs × n_vars = 30819 × 108660
    obs: 'dataset'
    var: 'gene_id', 'gene_name', 'gene_type', 'transcript_name', 'transcript_type', 'Chromosome', 'Start', 'End'